# Tensor Model-Parallel Group
- 张量并行（Tensor Parallelism, TP）是一种“层内模型并行”策略，它的核心思想是将单个 Transformer 层内的张量（如权重矩阵）切分并分配到多个 GPU 上，从而能够训练那些单卡显存无法容纳的超大模型。

在 Megatron 的 3D 并行（张量并行 TP + 流水线并行 PP + 数据并行 DP）中，'tp' 代表张量并行。
- 它的作用是将所有的 GPU 卡（World）划分成若干个组。<font color='red'>每个组内的 GPU 共同协作，负责切分并计算同一个模型层。</font>

## 💡 举个例子方便理解
假设你一共有 16 张 GPU 卡（编号 g0 到 g15），设置了 张量并行度 TP=4，流水线并行度 PP=2，数据并行度 DP=2。
- 这段代码执行时，decoder_rank_generator.get_ranks('tp') 会生成 4 个张量并行组（因为总共有 16 / 4 = 4 个 TP 组）：
- TP组 0 (ranks)：[g0, g1, g2, g3]
- TP组 1 (ranks)：[g4, g5, g6, g7]
- TP组 2 (ranks)：[g8, g9, g10, g11]
- TP组 3 (ranks)：[g12, g13, g14, g15]

<font color='red'>如果当前运行代码的是 g1 号卡：</font>
1. 循环第一次拿到 [g0, g1, g2, g3]。
2. 创建这个组的通信对象 group。
3. 发现 g1 在这个列表里，于是 g1 认领这个组，将 _TENSOR_MODEL_PARALLEL_GROUP 指向它。
4. 后续循环拿到的其他组都不包含 g1，全部跳过。

最终，g1 知道自己属于第一个张量并行组，后续它只需要和组内的 g0, g2, g3 进行模型层内部的切分计算与通信，而与 g4, g8 等卡完全隔离（它们属于其他 TP 组或 DP 组）。

In [ ]:
    '''
    张量 模型并行组：
    
    '''
    # Build the tensor model-parallel groups.
    global _TENSOR_MODEL_PARALLEL_GROUP
    global _TENSOR_MODEL_PARALLEL_GLOBAL_RANKS
    assert (
        _TENSOR_MODEL_PARALLEL_GROUP is None
    ), 'tensor model parallel group is already initialized'
    for ranks in decoder_rank_generator.get_ranks('tp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp", nccl_comm_cfgs),
            group_desc="TENSOR_MODEL_PARALLEL_GROUP",
        )
        if rank in ranks:
            _TENSOR_MODEL_PARALLEL_GROUP = group
            _TENSOR_MODEL_PARALLEL_GLOBAL_RANKS = ranks

## 每张卡（每个进程）都必须执行 create_group 来创建通信组对象

- 每张卡（每个进程）都必须执行 create_group 来创建通信组对象。这是分布式训练中的一个核心机制。
- 在 PyTorch 的分布式通信库（torch.distributed）中，创建一个新的进程组（Process Group）属于集体通信操作（Collective Communication）。
  - 这意味着，<font color='red'>当你定义了一个包含特定 GPU 卡号（ranks）的组时，该组内的所有卡必须在同一时刻调用创建函数，才能成功建立起它们之间的通信通道。</font>

为了帮你彻底理清这个逻辑，我们可以把这段代码的执行过程拆解为“全员参与”和“个人认领”两个层面：

### 1. 全员参与：create_group 是集体动作
decoder_rank_generator.get_ranks('tp') 会生成一系列包含不同卡号的列表（比如 [[0, 1], [2, 3], [4, 5]]）。当代码进入 for ranks in ... 循环时：
- <font color='red'>同步等待</font>：<font color='green'>假设当前循环到了 ranks = [0, 1]，此时 GPU 0 和 GPU 1 必须同时执行 create_group([0, 1], ...)。</font>如果只有 GPU 0 调用了这个函数，而 GPU 1 没调用（或者去执行别的代码了），GPU 0 就会一直卡在这里死等，最终导致整个训练程序超时崩溃。
- 建立通道：create_group 的本质是告诉底层的通信后端（如 NCCL）：“嘿，0号卡和1号卡现在要组成一个小分队，请给它们开辟一条专属的通信通道”。这条通道必须由小分队的双方共同确认建立。


### 2. 个人认领：if rank in ranks 是个人动作
虽然每张卡都要参与创建组的动作，但每张卡只需要保存属于自己的那个组。
- 认领归属：继续上面的例子，当创建 [0, 1] 这个组时，GPU 0 和 GPU 1 都会执行 create_group。紧接着，if rank in ranks 这个判断条件会同时在这两张卡上生效：
  - GPU 0 发现自己（rank=0）在 [0, 1] 里，于是把刚创建好的组对象存入自己的全局变量 _TENSOR_MODEL_PARALLEL_GROUP。
  - GPU 1 也发现自己（rank=1）在 [0, 1] 里，同样把这个组对象存入自己的全局变量。
- 旁观路过：当循环进行到下一轮 ranks = [2, 3] 时，GPU 0 和 GPU 1 依然要参与 create_group([2, 3], ...) 的创建过程（配合 GPU 2 和 3 完成建组），但在随后的 if rank in ranks 判断中，因为 0 和 1 不在 [2, 3] 里，它们就会直接跳过，不保存这个与己无关的组对象。

### 💡 举个通俗的例子
这就好比在学校里分学习小组：
- create_group（全员参与）：老师点名说“现在 A 组的同学（小明和小红）互相加一下微信”。这时，小明和小红都必须拿出手机扫码，这个“加好友”的动作才能成功。如果只有小明拿手机，小红在旁边发呆，好友是加不上的。
- if rank in ranks（个人认领）：加完好友后，小明和小红会把对方存进自己的通讯录，备注为“A组搭档”。而旁边 B 组的小刚和小丽虽然也在同一时间加了好友，但小明不需要把他们存进自己的“A组搭档”通讯录里。

所以，代码中 for 循环里的 create_group 是让所有卡配合彼此完成所有小组的搭建，而 if 判断是让每张卡认清自己到底属于哪个小组。